# Phase 7a — Dense Embedding Generation & ChromaDB Storage  (v4.0)

This notebook encodes the OpenSanctions corpus into dense vector
embeddings using two sentence-transformer models, then stores the
vectors in separate ChromaDB collections for downstream retrieval.

## What's new in v4.0

- **Enriched `build_doc_text`** — now reads `metadata` (country → full name,
  programId → readable label, datasets → source names) and `identifiers`
  (IMO numbers, call signs, registration numbers)
- **Before:** caption + schema + text_blob fallback (token soup)
- **After:** clean English sentences with country, sanctions program, and
  data-source context — dramatically better for semantic search
- **Enriched ChromaDB metadata** — country and program stored alongside
  caption/schema so search results show them

## Models

| Key | HF Name | Dim | Size | Notes |
|-----|---------|-----|------|-------|
| minilm | all-MiniLM-L6-v2 | 384 | 80 MB | Fast baseline |
| bge-m3 | BAAI/bge-m3 | 1024 | 2.2 GB | Multilingual, long-context |

## Pipeline

1. **Configure** — HuggingFace token (optional) and corpus size limit
2. **Auto-detect** project paths (no hardcoded paths)
3. **Load** the JSONL corpus (full or limited to N rows)
4. **Encode** every entity with **both** models, caching `.npy` to `models/`
5. **Store** each model's vectors in its own ChromaDB collection

## Naming Convention

All output files include a **row-count tag** and **creation date**:

| Example filename | Meaning |
|------------------|---------|
| `doc_embeddings_minilm_50K_20260407.npy` | minilm, 50 000 rows, created 7 Apr 2026 |
| `doc_embeddings_bge_m3_FULL_20260407.npy` | bge-m3, full corpus, created 7 Apr 2026 |
| `opensanctions_minilm_50K_20260407` | ChromaDB collection, same convention |


**CLI (recommended):** Run the same pipeline from the repo root without this notebook:

- `python scripts/build_dense_embeddings.py --help`
- Thin entrypoints: `scripts/encode_dense_minilm.py`, `scripts/encode_dense_bge_m3.py` (delegate to the script with `--model` fixed).

Corpus text is read from the `embedding_text` field on each JSONL line (see preprocessing pipeline).

---
## 0 · Emergency Reset (activate only if starting from zero)

Uncomment and run the cell below **only** to delete all cached
embeddings and ChromaDB collections.

In [3]:
# ---- UNCOMMENT TO RESET ----
# from pathlib import Path
# import chromadb
#
# _root = Path.cwd().resolve()
# _models = _root / 'models'
# _chroma = _root / 'chroma_db'
#
# # Delete all embedding files
# if _models.exists():
#     for f in _models.glob('doc_embeddings_*'):
#         f.unlink(); print(f"Deleted {f}")
#     for f in _models.glob('doc_ids_*'):
#         f.unlink(); print(f"Deleted {f}")
#
# # Delete all opensanctions collections
# if _chroma.exists():
#     client = chromadb.PersistentClient(path=str(_chroma))
#     for c in client.list_collections():
#         cname = c.name if hasattr(c, 'name') else str(c)
#         if cname.startswith('opensanctions_'):
#             client.delete_collection(cname)
#             print(f"Deleted collection {cname}")

---
## 1 · Configuration

Set two things before anything else:
1. **HuggingFace token** — needed for gated models; press Enter to skip
2. **Corpus row limit** — press Enter for all rows, or type a number

In [5]:
import os
import getpass
from datetime import date

# ---- 1. HuggingFace Token ----
_hf_token = getpass.getpass("Enter HuggingFace token (press Enter to skip): ").strip()

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    if len(_hf_token) > 8:
        masked = _hf_token[:4] + "*" * (len(_hf_token) - 8) + _hf_token[-4:]
    else:
        masked = "****"
    print(f"HF_TOKEN set: {masked}")
else:
    print("No HF_TOKEN provided — public models only (sufficient for minilm & bge-m3).")

del _hf_token

# ---- 2. Corpus Row Limit ----
_limit_input = input("Max corpus rows to load (Enter = all): ").strip()

if _limit_input:
    try:
        CORPUS_LIMIT = int(_limit_input)
        print(f"Corpus limited to first {CORPUS_LIMIT:,} rows.")
    except ValueError:
        CORPUS_LIMIT = None
        print(f"Could not parse '{_limit_input}' — loading full corpus.")
else:
    CORPUS_LIMIT = None
    print("Loading full corpus (no limit).")

# ---- 3. Build the naming suffix: e.g. "50K_20260407" or "FULL_20260407" ----
_date_tag = date.today().strftime("%Y%m%d")

if CORPUS_LIMIT is not None:
    if CORPUS_LIMIT >= 1_000_000:
        _size_tag = f"{CORPUS_LIMIT // 1_000_000}M"
    elif CORPUS_LIMIT >= 1_000:
        _size_tag = f"{CORPUS_LIMIT // 1_000}K"
    else:
        _size_tag = str(CORPUS_LIMIT)
else:
    _size_tag = "FULL"

RUN_TAG = f"{_size_tag}_{_date_tag}"
print(f"Run tag      : {RUN_TAG}")
print(f"  (files will be named e.g. doc_embeddings_minilm_{RUN_TAG}.npy)")

Enter HuggingFace token (press Enter to skip):  ········


HF_TOKEN set: hf_E*****************************cVGJ


Max corpus rows to load (Enter = all):  


Loading full corpus (no limit).
Run tag      : FULL_20260407
  (files will be named e.g. doc_embeddings_minilm_FULL_20260407.npy)


---
## 2 · Imports and Setup

In [7]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import time
import warnings
import torch
import shutil
import psutil
import logging
from pathlib import Path

from tqdm import tqdm
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb
import pycountry

warnings.filterwarnings('ignore')
logging.getLogger("safetensors").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)


def mem_report(tag=""):
    """Print current RAM usage."""
    proc = psutil.Process()
    rss = proc.memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    print(f"  [MEM {tag}] RSS={rss:.2f} GB | Available={avail:.2f} GB")


mem_report("after imports")
print("All imports loaded.")


  [MEM after imports] RSS=0.79 GB | Available=5.76 GB
All imports loaded.


In [8]:
# ======================================================================
# Auto-detect project paths
# ======================================================================

def find_corpus():
    """Search upward from CWD for documents.jsonl in common layouts."""
    patterns = [
        'processed/full/documents.jsonl',
        'data/processed/full/documents.jsonl',
        'IR_project/processed/full/documents.jsonl',
    ]
    current = Path.cwd().resolve()
    for _ in range(6):
        for pat in patterns:
            candidate = current / pat
            if candidate.exists():
                return candidate.resolve()
        current = current.parent
    return None

CORPUS_PATH = find_corpus()

if CORPUS_PATH is None:
    raise FileNotFoundError(
        "Could not find 'documents.jsonl' anywhere in the project tree.\n"
        f"  Searched upward from: {Path.cwd()}\n"
        "  Set CORPUS_PATH manually if your layout is non-standard:\n"
        "    CORPUS_PATH = Path(r'C:\\...\\documents.jsonl')"
    )

PROJECT_ROOT = CORPUS_PATH.parent.parent.parent
MODELS_DIR   = PROJECT_ROOT / 'models'
CHROMA_DIR   = PROJECT_ROOT / 'chroma_db'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# ---- Device ----
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected : {gpu_name}  ({vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU — falling back to CPU (encoding will be slow).")

# ======================================================================
# Model Registry
# ======================================================================
MODEL_REGISTRY = {
    'minilm': {
        'name':               'all-MiniLM-L6-v2',
        'dim':                384,
        'batch_size':         512 if DEVICE == 'cuda' else 64,
        'chunk_size':         100_000,
        'trust_remote_code':  False,
        'notes':              'Lightweight baseline — fast, good quality',
    },
    'bge-m3': {
        'name':               'BAAI/bge-m3',
        'dim':                1024,
        'batch_size':         128 if DEVICE == 'cuda' else 16,
        'chunk_size':         50_000,
        'trust_remote_code':  False,
        'notes':              'Multilingual M3 — dense + sparse + ColBERT',
    },
}

# ======================================================================
# Adaptive ChromaDB batch size
# ======================================================================
# The bottleneck is .tolist() converting numpy arrays to Python lists.
# RAM cost per batch ≈ batch_size × dim × 8 bytes (Python float overhead).
#
# Formula: target ~50 MB per batch → batch = 50MB / (dim × 8)
# Then capped at ChromaDB's internal max_batch_size (typically ~5,461
# derived from HNSW sync_threshold). We use 5,000 as a safe hard cap.
# ======================================================================

CHROMA_MAX_BATCH = 5_000   # ChromaDB internal limit is ~5,461; stay under

def compute_chroma_batch_size(dim, available_ram_gb=None):
    """Compute optimal ChromaDB insert batch size for a given embedding dim."""
    if available_ram_gb is None:
        available_ram_gb = psutil.virtual_memory().available / 1e9

    # Target ~50 MB per batch when RAM > 4 GB, ~20 MB when tight
    target_mb = 50 if available_ram_gb > 4 else 20
    target_bytes = target_mb * 1e6

    batch = int(target_bytes / (dim * 8))

    # Clamp: floor 1,000 — ceiling min(10,000, ChromaDB hard limit)
    batch = max(1_000, min(batch, CHROMA_MAX_BATCH))

    return batch

# ======================================================================
# Build resolved configs with RUN_TAG in filenames
# ======================================================================
SELECTED_MODELS = list(MODEL_REGISTRY.keys())
SELECTED_CONFIGS = {}

for model_key in SELECTED_MODELS:
    cfg = MODEL_REGISTRY[model_key]
    suffix = model_key.replace('-', '_')

    chroma_batch = compute_chroma_batch_size(cfg['dim'])

    SELECTED_CONFIGS[model_key] = {
        **cfg,
        'key':               model_key,
        'embedding_cache':   MODELS_DIR / f'doc_embeddings_{suffix}_{RUN_TAG}.npy',
        'doc_ids_cache':     MODELS_DIR / f'doc_ids_{suffix}_{RUN_TAG}.json',
        'chroma_collection': f'opensanctions_{suffix}_{RUN_TAG}',
        'chroma_batch_size': chroma_batch,
    }

# ---- Summary ----
print(f"Project root : {PROJECT_ROOT}")
print(f"Corpus       : {CORPUS_PATH}")
print(f"Models dir   : {MODELS_DIR}")
print(f"ChromaDB     : {CHROMA_DIR}")
print(f"Device       : {DEVICE}")
print(f"Corpus limit : {CORPUS_LIMIT if CORPUS_LIMIT else 'ALL'}")
print(f"Run tag      : {RUN_TAG}")
print(f"Chroma cap   : {CHROMA_MAX_BATCH:,} (ChromaDB internal limit)")
print()
for mk in SELECTED_MODELS:
    sc = SELECTED_CONFIGS[mk]
    cached = "CACHED" if sc['embedding_cache'].exists() else "will encode"
    print(f"  {mk:10s}  dim={sc['dim']}  encode_batch={sc['batch_size']}  "
          f"chroma_batch={sc['chroma_batch_size']}  [{cached}]")
    print(f"             {sc['notes']}")
    print(f"             npy: {sc['embedding_cache'].name}")
    print(f"             col: {sc['chroma_collection']}")

GPU detected : NVIDIA GeForce RTX 4080  (17.2 GB VRAM)
Project root : C:\Users\marek\OneDrive\IR_2\IR_project
Corpus       : C:\Users\marek\OneDrive\IR_2\IR_project\processed\full\documents.jsonl
Models dir   : C:\Users\marek\OneDrive\IR_2\IR_project\models
ChromaDB     : C:\Users\marek\OneDrive\IR_2\IR_project\chroma_db
Device       : cuda
Corpus limit : ALL
Run tag      : FULL_20260407
Chroma cap   : 5,000 (ChromaDB internal limit)

  minilm      dim=384  encode_batch=512  chroma_batch=5000  [will encode]
             Lightweight baseline — fast, good quality
             npy: doc_embeddings_minilm_FULL_20260407.npy
             col: opensanctions_minilm_FULL_20260407
  bge-m3      dim=1024  encode_batch=128  chroma_batch=5000  [will encode]
             Multilingual M3 — dense + sparse + ColBERT
             npy: doc_embeddings_bge_m3_FULL_20260407.npy
             col: opensanctions_bge_m3_FULL_20260407


---
## 3 · Load Corpus

In [10]:
def load_corpus(path, limit=None):
    """Load JSONL corpus, optionally limited to first `limit` rows."""
    if path is None:
        raise FileNotFoundError("CORPUS_PATH is None — see path detection cell above.")
    docs = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if line:
                docs.append(json.loads(line))
    return docs

corpus = load_corpus(CORPUS_PATH, limit=CORPUS_LIMIT)
n_docs_actual = len(corpus)
print(f"Corpus: {n_docs_actual:,} documents" +
      (f" (limited from full file)" if CORPUS_LIMIT else ""))

if CORPUS_LIMIT and n_docs_actual < CORPUS_LIMIT:
    print(f"  Note: requested {CORPUS_LIMIT:,} but file only has {n_docs_actual:,} rows.")
    print(f"  RUN_TAG remains '{RUN_TAG}' (based on requested limit).")

mem_report("after corpus load")

Corpus: 1,248,365 documents
  [MEM after corpus load] RSS=5.91 GB | Available=1.40 GB


---
## 4 · Text Builder for Embeddings (v4.0 — enriched)

**Key change from v3.0:** The text builder now reads `metadata` and
`identifiers` from every record, converting:
- Country codes → full names (e.g. `lb` → `Lebanon`)
- Program IDs → readable labels (e.g. `US-NARCO` → `U.S. Foreign Narcotics Kingpin Designation Act`)
- Dataset slugs → source names (e.g. `us_ofac_sdn` → `U.S. OFAC SDN List`)
- Identifier fields (IMO, call sign, registration numbers)

The `text_blob` fallback is only used when metadata is completely empty.


In [12]:
# ======================================================================
# ISO-3166-1 alpha-2 → full country name (covers ~250 codes)
# ======================================================================

import pycountry

_COUNTRY_CACHE = {}

def _country_name(code):
    """Convert 2-letter ISO code to full country name, with cache."""
    code = code.strip().upper()
    if code in _COUNTRY_CACHE:
        return _COUNTRY_CACHE[code]
    try:
        name = pycountry.countries.get(alpha_2=code).name
    except (AttributeError, LookupError):
        # Fallback for non-standard codes
        _EXTRA = {
            'XK': 'Kosovo', 'TW': 'Taiwan', 'PS': 'Palestine',
            'AN': 'Netherlands Antilles', 'CS': 'Serbia and Montenegro',
            'SU': 'Soviet Union', 'YU': 'Yugoslavia', 'XX': 'Unknown',
        }
        name = _EXTRA.get(code, code)
    _COUNTRY_CACHE[code] = name
    return name


# ======================================================================
# Program-ID prefix → human-readable programme label
# ======================================================================

_PROGRAM_LABELS = {
    'US-OFAC':     'U.S. OFAC sanctions',
    'US-NARCO':    'U.S. Foreign Narcotics Kingpin Designation Act',
    'US-GLOMAG':   'U.S. Global Magnitsky sanctions',
    'US-BIS':      'U.S. Bureau of Industry and Security',
    'US-BIS-DPL':  'U.S. BIS Denied Persons List',
    'US-BIS-EL':   'U.S. BIS Entity List',
    'US-SDGT':     'U.S. Specially Designated Global Terrorist',
    'US-SDN':      'U.S. Specially Designated Nationals',
    'US-SAM':      'U.S. SAM Exclusions',
    'US-HHS-OIG':  'U.S. HHS Office of Inspector General exclusions',
    'EU-UKR':      'EU Ukraine-related sanctions',
    'EU-FSF':      'EU Financial Sanctions',
    'UA-SA1644':   'Ukraine NSDC sanctions',
    'CA-SEMA':     'Canada Special Economic Measures Act',
    'INTERPOL-RN': 'Interpol Red Notice',
    'SECO':        'Swiss SECO sanctions',
    'AU-SANCTIONS': 'Australian sanctions',
    'GB-HMT':      'UK HM Treasury sanctions',
}

def _program_label(prog_id):
    """Convert a programId like 'US-NARCO' to a readable label."""
    pid = prog_id.strip().upper()
    if pid in _PROGRAM_LABELS:
        return _PROGRAM_LABELS[pid]
    # Try prefix match (e.g. 'US-BIS-FOO' matches 'US-BIS')
    for prefix in sorted(_PROGRAM_LABELS, key=len, reverse=True):
        if pid.startswith(prefix):
            return _PROGRAM_LABELS[prefix]
    # Fallback: make it readable
    return pid.replace('-', ' ').replace('_', ' ')


# ======================================================================
# Dataset slug → human-readable source label
# ======================================================================

_DATASET_LABELS = {
    'us_ofac_sdn':                 'U.S. OFAC SDN List',
    'us_ofac_cons':                'U.S. OFAC Consolidated List',
    'us_sam_exclusions':           'U.S. SAM Exclusions',
    'us_trade_csl':                'U.S. Consolidated Screening List',
    'us_bis_denied':               'U.S. BIS Denied Persons',
    'us_bis_entity':               'U.S. BIS Entity List',
    'eu_journal_sanctions':        'EU Official Journal sanctions',
    'eu_fsf':                      'EU Financial Sanctions',
    'gb_hmt_sanctions':            'UK HM Treasury sanctions',
    'un_sc_sanctions':             'UN Security Council sanctions',
    'interpol_red_notices':        'Interpol Red Notices',
    'ch_seco_sanctions':           'Swiss SECO sanctions',
    'opencorporates':              'OpenCorporates registry',
    'ext_us_ofac_press_releases':  'OFAC press releases',
    'ua_nsdc_sanctions':           'Ukraine NSDC sanctions',
    'ca_sema_sanctions':           'Canada SEMA sanctions',
    'au_dfat_sanctions':           'Australian DFAT sanctions',
}

def _dataset_label(ds):
    """Convert dataset slug to readable label."""
    return _DATASET_LABELS.get(ds, ds.replace('_', ' ').title())


# ======================================================================
# build_doc_text v2 — enriched with metadata, identifiers, country names
# ======================================================================

def build_doc_text(doc):
    """
    Build a natural-language text representation of an entity for embedding.

    v4.0 changes:
    - Pulls from metadata.country (ISO → full name), metadata.programId,
      metadata.datasets — these were ignored in v3.0
    - Adds identifier types (IMO, call sign, registration numbers)
    - Falls back to text_blob only when everything else is sparse
    - Produces clean English sentences that sentence-transformers can
      actually encode meaningfully
    """
    parts = []

    # ---- 1. Caption and schema (always present) ----
    caption = doc.get('caption', '')
    schema  = doc.get('schema', '')
    if caption:
        parts.append(f"{caption}.")
    if schema:
        parts.append(f"Type: {schema}.")

    # ---- 2. Metadata: country, program, datasets ----
    meta = doc.get('metadata', {})

    countries_raw = meta.get('country', [])
    if countries_raw and isinstance(countries_raw, list):
        country_names = [_country_name(c) for c in countries_raw[:5]]
        parts.append(f"Country: {', '.join(country_names)}.")

    programs_raw = meta.get('programId', [])
    if programs_raw and isinstance(programs_raw, list):
        # Deduplicate while preserving order
        seen = set()
        prog_labels = []
        for p in programs_raw[:4]:
            lbl = _program_label(p)
            if lbl not in seen:
                prog_labels.append(lbl)
                seen.add(lbl)
        if prog_labels:
            parts.append(f"Sanctioned under: {'; '.join(prog_labels)}.")

    datasets_raw = meta.get('datasets', [])
    if datasets_raw and isinstance(datasets_raw, list):
        # Pick up to 3 most informative dataset labels
        ds_labels = []
        seen_ds = set()
        for ds in datasets_raw[:6]:
            lbl = _dataset_label(ds)
            if lbl not in seen_ds:
                ds_labels.append(lbl)
                seen_ds.add(lbl)
        if ds_labels:
            parts.append(f"Listed in: {'; '.join(ds_labels[:3])}.")

    # ---- 3. Identifiers (vessel IMO, call signs, registration #s) ----
    idents = doc.get('identifiers', {})
    if isinstance(idents, dict):
        id_parts = []
        if idents.get('imoNumber'):
            imo_vals = idents['imoNumber']
            if isinstance(imo_vals, list):
                id_parts.append(f"IMO number: {', '.join(str(v) for v in imo_vals[:2])}")
        if idents.get('callSign'):
            cs_vals = idents['callSign']
            if isinstance(cs_vals, list):
                id_parts.append(f"Call sign: {', '.join(str(v) for v in cs_vals[:2])}")
        if idents.get('registrationNumber'):
            reg_vals = idents['registrationNumber']
            if isinstance(reg_vals, list):
                id_parts.append(f"Registration: {', '.join(str(v) for v in reg_vals[:2])}")
        if idents.get('innCode'):
            inn_vals = idents['innCode']
            if isinstance(inn_vals, list):
                id_parts.append(f"INN code: {', '.join(str(v) for v in inn_vals[:2])}")
        if id_parts:
            parts.append(' '.join(id_parts) + '.')

    # ---- 4. Fallback: text_blob (only if still sparse) ----
    # "Sparse" = only caption + schema, no country/program/identifiers added
    if len(parts) <= 2:
        blob = doc.get('text_blob', '')
        if blob:
            parts.append(blob[:500])

    return ' '.join(parts)


# ---- Preview ----
for i in range(min(5, len(corpus))):
    text = build_doc_text(corpus[i])
    print(f"Doc {i}: {corpus[i].get('caption', 'N/A')}")
    print(f"  -> {text[:300]}...")
    print()



Doc 0: Myanmar Yatai International Holding Group Co., LTD.
  -> Myanmar Yatai International Holding Group Co., LTD.. Type: Company. Country: Myanmar. Sanctioned under: U.S. Global Magnitsky sanctions. Listed in: U.S. SAM Exclusions; OpenCorporates registry; OFAC press releases. Registration: 103919088....

Doc 1: Товариство з обмеженою відповідальністю "Зелінський Групп"
  -> Товариство з обмеженою відповідальністю "Зелінський Групп". Type: Company. Sanctioned under: Ukraine NSDC sanctions. Listed in: Ext Ru Egrul; Ukraine NSDC sanctions. INN code: 7725491052....

Doc 2: SANAVBARI NIKITENKO
  -> SANAVBARI NIKITENKO. Type: Person. Country: Tajikistan. Sanctioned under: Interpol Red Notice. Listed in: Interpol Red Notices....

Doc 3: Open Joint Stock Company “Elektrostal Chemical and Mechanical Plant named after N.D. Zelinsky”
  -> Open Joint Stock Company “Elektrostal Chemical and Mechanical Plant named after N.D. Zelinsky”. Type: Company. Country: Russian Federation. Sanctioned under: 

---
## 5 · Encode Corpus with Both Models

Each model encodes sequentially: load → encode in chunks (streamed to
disk via memory-mapped `.npy` in `models/`) → free GPU → next model.

Cached embeddings are reused automatically on re-runs.

**Speed optimisations:** fp16 inference, large batches, chunked disk
writes, periodic GC every 4 chunks.

In [14]:
# Build doc_texts and doc_ids once (shared across both models)
print("Building document text representations...")
doc_texts = [build_doc_text(doc)
             for doc in tqdm(corpus, desc="Build doc text", unit="doc")]
doc_ids   = [doc.get('doc_id', doc.get('id', f'doc_{i}'))
             for i, doc in enumerate(corpus)]
n_docs    = len(doc_texts)
print(f"  {n_docs:,} documents prepared for encoding.")

# Pre-encoding cleanup
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

model_embeddings = {}
mem_report("before encoding")

for model_idx, model_key in enumerate(SELECTED_MODELS, 1):
    sc = SELECTED_CONFIGS[model_key]
    print(f"\n{'='*60}")
    print(f"  [{model_idx}/{len(SELECTED_MODELS)}] {model_key} -> {sc['name']}")
    print(f"  Dim: {sc['dim']}  Batch: {sc['batch_size']}  Chunk: {sc['chunk_size']:,}")
    print(f"  Cache: {sc['embedding_cache'].name}")
    print(f"{'='*60}")

    # ---- Check cache ----
    if sc['embedding_cache'].exists() and sc['doc_ids_cache'].exists():
        cached_ids = json.loads(sc['doc_ids_cache'].read_text())
        if cached_ids == doc_ids:
            print(f"  Loading cached embeddings (memory-mapped)...")
            model_embeddings[model_key] = np.load(str(sc['embedding_cache']), mmap_mode='r')
            print(f"  Loaded {model_embeddings[model_key].shape} from cache — skipping encoding.")
            continue
        else:
            print("  Cache doc_ids mismatch — will re-encode.")
            sc['embedding_cache'].unlink(missing_ok=True)
            sc['doc_ids_cache'].unlink(missing_ok=True)

    # ---- Load model ----
    print(f"  Loading model '{sc['name']}' on {DEVICE.upper()}...")
    load_kwargs = dict(device=DEVICE)
    if sc['trust_remote_code']:
        load_kwargs['trust_remote_code'] = True
    _hf_token = os.environ.get("HF_TOKEN", None)
    if _hf_token:
        load_kwargs['token'] = _hf_token
    _model = SentenceTransformer(sc['name'], **load_kwargs)

    if DEVICE == 'cuda':
        _model = _model.half()
        print("  Using fp16 inference for speed.")

    actual_dim = _model.get_sentence_embedding_dimension()
    print(f"  Model loaded on : {next(_model.parameters()).device}")
    print(f"  Embedding dim   : {actual_dim}")
    if actual_dim != sc['dim']:
        print(f"  WARNING: registry says {sc['dim']} but model gives {actual_dim}. Updating.")
        sc['dim'] = actual_dim

    # ---- Pre-allocate on disk (memory-mapped) ----
    cache_path = str(sc['embedding_cache'].resolve())
    tmp_path   = cache_path.replace('.npy', '_tmp.npy')

    if Path(tmp_path).exists():
        Path(tmp_path).unlink()

    fp = np.lib.format.open_memmap(
        tmp_path, mode='w+', dtype=np.float32,
        shape=(n_docs, actual_dim),
    )
    print(f"  Pre-allocated {n_docs:,} x {actual_dim} float32 on disk")

    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    print(f"  Encoding {n_docs:,} documents (batch={sc['batch_size']})...")
    start = time.time()
    pbar = tqdm(total=n_docs, desc=f"Encode [{model_key}]", unit="doc")
    chunk_count = 0

    for chunk_start in range(0, n_docs, sc['chunk_size']):
        chunk_end = min(chunk_start + sc['chunk_size'], n_docs)
        chunk = doc_texts[chunk_start:chunk_end]
        emb = _model.encode(
            chunk,
            batch_size=sc['batch_size'],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        fp[chunk_start:chunk_end] = emb
        del emb, chunk
        pbar.update(chunk_end - chunk_start)
        chunk_count += 1

        if chunk_count % 4 == 0:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            gc.collect()

    pbar.close()
    fp.flush()
    del fp
    gc.collect()

    elapsed = time.time() - start
    print(f"  Encoding complete in {elapsed:.1f}s  ({n_docs/elapsed:,.0f} docs/sec)")

    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        print(f"  Peak GPU memory: {peak_mb:.0f} MB")

    shutil.move(tmp_path, cache_path)
    sc['doc_ids_cache'].write_text(json.dumps(doc_ids))
    print(f"  Saved -> {sc['embedding_cache'].name}")

    model_embeddings[model_key] = np.load(cache_path, mmap_mode='r')

    del _model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f"  Released {model_key} model from GPU.")
    mem_report(f"after {model_key} encoding")

print(f"\n{'='*60}")
print(f"Both models encoded:")
for mk, emb in model_embeddings.items():
    print(f"  * {mk:10s} -> {emb.shape}  ({SELECTED_CONFIGS[mk]['embedding_cache'].name})")
print(f"{'='*60}")

Building document text representations...


Build doc text: 100%|███████████████████████████████████████████████████| 1248365/1248365 [00:04<00:00, 306553.77doc/s]


  1,248,365 documents prepared for encoding.
  [MEM before encoding] RSS=6.15 GB | Available=1.17 GB

  [1/2] minilm -> all-MiniLM-L6-v2
  Dim: 384  Batch: 512  Chunk: 100,000
  Cache: doc_embeddings_minilm_FULL_20260407.npy
  Loading model 'all-MiniLM-L6-v2' on CUDA...
  Using fp16 inference for speed.
  Model loaded on : cuda:0
  Embedding dim   : 384
  Pre-allocated 1,248,365 x 384 float32 on disk
  Encoding 1,248,365 documents (batch=512)...


Encode [minilm]: 100%|████████████████████████████████████████████████████| 1248365/1248365 [03:36<00:00, 5771.08doc/s]


  Encoding complete in 251.1s  (4,972 docs/sec)
  Peak GPU memory: 1217 MB
  Saved -> doc_embeddings_minilm_FULL_20260407.npy
  Released minilm model from GPU.
  [MEM after minilm encoding] RSS=6.62 GB | Available=4.13 GB

  [2/2] bge-m3 -> BAAI/bge-m3
  Dim: 1024  Batch: 128  Chunk: 50,000
  Cache: doc_embeddings_bge_m3_FULL_20260407.npy
  Loading model 'BAAI/bge-m3' on CUDA...
  Using fp16 inference for speed.
  Model loaded on : cuda:0
  Embedding dim   : 1024
  Pre-allocated 1,248,365 x 1024 float32 on disk
  Encoding 1,248,365 documents (batch=128)...


Encode [bge-m3]: 100%|████████████████████████████████████████████████████| 1248365/1248365 [09:27<00:00, 2197.92doc/s]


  Encoding complete in 690.4s  (1,808 docs/sec)
  Peak GPU memory: 1839 MB
  Saved -> doc_embeddings_bge_m3_FULL_20260407.npy
  Released bge-m3 model from GPU.
  [MEM after bge-m3 encoding] RSS=5.90 GB | Available=6.36 GB

Both models encoded:
  * minilm     -> (1248365, 384)  (doc_embeddings_minilm_FULL_20260407.npy)
  * bge-m3     -> (1248365, 1024)  (doc_embeddings_bge_m3_FULL_20260407.npy)


---
## 6 · Store in ChromaDB

Each model gets its own ChromaDB collection with pre-computed embeddings.
No embedding function is loaded at insert time (saves RAM).

**v4.0:** ChromaDB metadata now includes `country` and `program` fields
alongside `caption` and `schema`, so search results display richer context.

**Batch size tuning:** Computed per model based on dimension and
available RAM, then hard-capped at 5,000 to stay within ChromaDB's
internal `max_batch_size` limit (~5,461).

| Model | Dim | Chroma batch | RAM/batch |
|-------|-----|-------------|-----------|
| minilm | 384 | 5,000 (capped) | ~15 MB |
| bge-m3 | 1024 | 5,000 (capped) | ~41 MB |


In [16]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

model_collections = {}

for model_key in SELECTED_MODELS:
    sc = SELECTED_CONFIGS[model_key]
    coll_name = sc['chroma_collection']
    coll = chroma_client.get_or_create_collection(
        name=coll_name,
        metadata={
            "hnsw:space":           "cosine",
            "hnsw:M":               16,
            "hnsw:construction_ef": 100,
            "hnsw:batch_size":      5000,
            "hnsw:sync_threshold":  5000,
        },
    )
    model_collections[model_key] = coll
    print(f"  {model_key:10s} -> '{coll_name}': {coll.count():,} docs on disk")

print(f"\nAll collections in {CHROMA_DIR}:")
for c in chroma_client.list_collections():
    cname = c.name if hasattr(c, 'name') else str(c)
    cnt = chroma_client.get_collection(cname).count()
    print(f"  * {cname}: {cnt:,} docs")

mem_report("after collection setup")


  minilm     -> 'opensanctions_minilm_FULL_20260407': 0 docs on disk
  bge-m3     -> 'opensanctions_bge_m3_FULL_20260407': 0 docs on disk

All collections in C:\Users\marek\OneDrive\IR_2\IR_project\chroma_db:
  * opensanctions_minilm_23K_20260407: 0 docs
  * opensanctions_bge_m3_23K_20260407: 0 docs
  * opensanctions_bge_m3_FULL_20260407: 0 docs
  * opensanctions_minilm_FULL_20260407: 0 docs
  * opensanctions_bge_m3_16K_20260407: 16,000 docs
  * opensanctions_minilm_16K_20260407: 16,000 docs
  * opensanctions_bge_m3: 50,000 docs
  * opensanctions_minilm_77K_20260407: 77,000 docs
  * opensanctions_bge_m3_77K_20260407: 77,000 docs
  * opensanctions_minilm: 50,000 docs
  [MEM after collection setup] RSS=6.94 GB | Available=5.27 GB


In [17]:
# Free doc_texts — not needed during insert
if 'doc_texts' in dir():
    del doc_texts
    gc.collect()
    print("Freed doc_texts from RAM.")

mem_report("before ChromaDB insert")

for model_key in SELECTED_MODELS:
    sc   = SELECTED_CONFIGS[model_key]
    coll = model_collections[model_key]
    emb  = model_embeddings[model_key]
    CHROMA_BATCH = sc['chroma_batch_size']

    if coll.count() == len(corpus):
        print(f"[{model_key}] ChromaDB already has {coll.count():,} docs — skipping.")
        continue

    if coll.count() > 0:
        print(f"[{model_key}] Count mismatch ({coll.count():,} vs {len(corpus):,}). Rebuilding...")
        chroma_client.delete_collection(sc['chroma_collection'])
        coll = chroma_client.get_or_create_collection(
            name=sc['chroma_collection'],
            metadata={
                "hnsw:space":           "cosine",
                "hnsw:M":               16,
                "hnsw:construction_ef": 100,
                "hnsw:batch_size":      5000,
                "hnsw:sync_threshold":  5000,
            },
        )
        model_collections[model_key] = coll

    print(f"[{model_key}] Inserting {len(corpus):,} docs (chroma_batch={CHROMA_BATCH})...")
    start = time.time()

    n_batches = (len(corpus) + CHROMA_BATCH - 1) // CHROMA_BATCH
    batch_iter = tqdm(
        range(0, len(corpus), CHROMA_BATCH),
        total=n_batches,
        desc=f"ChromaDB [{model_key}]",
        unit="batch",
    )

    for batch_num, i in enumerate(batch_iter):
        batch_end = min(i + CHROMA_BATCH, len(corpus))

        batch_meta = []
        for j in range(i, batch_end):
            doc = corpus[j]
            meta_raw = doc.get('metadata', {})
            countries = meta_raw.get('country', [])
            programs  = meta_raw.get('programId', [])
            batch_meta.append({
                'caption':  str(doc.get('caption', ''))[:200],
                'schema':   str(doc.get('schema', '')),
                'country':  ', '.join(str(c).upper() for c in (countries if isinstance(countries, list) else []))[:100],
                'program':  ', '.join(str(p) for p in (programs if isinstance(programs, list) else []))[:200],
            })

        batch_emb = np.ascontiguousarray(emb[i:batch_end]).tolist()

        coll.add(
            ids=doc_ids[i:batch_end],
            embeddings=batch_emb,
            metadatas=batch_meta,
        )

        del batch_emb, batch_meta

        if batch_num % 10 == 0:
            gc.collect()

        batch_iter.set_postfix(docs=f"{batch_end:,}")

    elapsed = time.time() - start
    print(f"[{model_key}] Done in {elapsed:.1f}s — {coll.count():,} documents stored")
    print(f"  Collection: {sc['chroma_collection']}")
    mem_report(f"after {model_key} insert")

# ---- Final cleanup ----
del model_embeddings
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print("\nFreed all embeddings from RAM.")
mem_report("final")


Freed doc_texts from RAM.
  [MEM before ChromaDB insert] RSS=6.73 GB | Available=5.46 GB
[minilm] Inserting 1,248,365 docs (chroma_batch=5000)...


ChromaDB [minilm]: 100%|██████████████████████████████████████████| 250/250 [25:56<00:00,  6.22s/batch, docs=1,248,365]


[minilm] Done in 1556.0s — 1,248,365 documents stored
  Collection: opensanctions_minilm_FULL_20260407
  [MEM after minilm insert] RSS=9.11 GB | Available=2.48 GB
[bge-m3] Inserting 1,248,365 docs (chroma_batch=5000)...


ChromaDB [bge-m3]: 100%|████████████████████████████████████████| 250/250 [1:07:01<00:00, 16.09s/batch, docs=1,248,365]


[bge-m3] Done in 4021.4s — 1,248,365 documents stored
  Collection: opensanctions_bge_m3_FULL_20260407
  [MEM after bge-m3 insert] RSS=11.29 GB | Available=1.50 GB

Freed all embeddings from RAM.
  [MEM final] RSS=11.28 GB | Available=1.51 GB


---
## 7 · Summary

In [19]:
print("=" * 60)
print("  EMBEDDING GENERATION & CHROMADB STORAGE COMPLETE")
print("=" * 60)
print()
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Corpus docs  : {len(corpus):,}")
print(f"  Run tag      : {RUN_TAG}")
print(f"  Device used  : {DEVICE}")
print()

for model_key in SELECTED_MODELS:
    sc   = SELECTED_CONFIGS[model_key]
    coll = model_collections[model_key]
    print(f"  Model        : {model_key} ({sc['name']})")
    print(f"  Dimension    : {sc['dim']}")
    print(f"  Embeddings   : {sc['embedding_cache'].name}")
    print(f"  Doc IDs      : {sc['doc_ids_cache'].name}")
    print(f"  Collection   : {sc['chroma_collection']} ({coll.count():,} docs)")
    print(f"  Chroma batch : {sc['chroma_batch_size']:,}")
    print()

print(f"  Models dir   : {MODELS_DIR}")
print(f"  ChromaDB dir : {CHROMA_DIR}")
print()
print("Both models ready for retrieval in downstream notebooks.")
print()
print("To use in Notebook 7b, set:")
print(f"  ACTIVE_MODEL = 'minilm'   # or 'bge-m3'")
print(f"  RUN_TAG      = '{RUN_TAG}'")

  EMBEDDING GENERATION & CHROMADB STORAGE COMPLETE

  Project root : C:\Users\marek\OneDrive\IR_2\IR_project
  Corpus docs  : 1,248,365
  Run tag      : FULL_20260407
  Device used  : cuda

  Model        : minilm (all-MiniLM-L6-v2)
  Dimension    : 384
  Embeddings   : doc_embeddings_minilm_FULL_20260407.npy
  Doc IDs      : doc_ids_minilm_FULL_20260407.json
  Collection   : opensanctions_minilm_FULL_20260407 (1,248,365 docs)
  Chroma batch : 5,000

  Model        : bge-m3 (BAAI/bge-m3)
  Dimension    : 1024
  Embeddings   : doc_embeddings_bge_m3_FULL_20260407.npy
  Doc IDs      : doc_ids_bge_m3_FULL_20260407.json
  Collection   : opensanctions_bge_m3_FULL_20260407 (1,248,365 docs)
  Chroma batch : 5,000

  Models dir   : C:\Users\marek\OneDrive\IR_2\IR_project\models
  ChromaDB dir : C:\Users\marek\OneDrive\IR_2\IR_project\chroma_db

Both models ready for retrieval in downstream notebooks.

To use in Notebook 7b, set:
  ACTIVE_MODEL = 'minilm'   # or 'bge-m3'
  RUN_TAG      = 'FULL_2